In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
import os

%matplotlib inline
sns.set_theme(style="whitegrid")

In [12]:
#Reads the .csv file
df = pd.read_csv(r"C:\Users\harri\OneDrive\Documents\DataScienceProjects\Retail_Demand_Forecasting_Project\data\demand_forecasting.csv")

#Prints the shape of the dataframe out
df.shape

(76000, 16)

In [13]:
#Shows the data types for the dataset
df.dtypes

Date                   object
Store ID               object
Product ID             object
Category               object
Region                 object
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Price                 float64
Discount                int64
Weather Condition      object
Promotion               int64
Competitor Pricing    float64
Seasonality            object
Epidemic                int64
Demand                  int64
dtype: object

In [14]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [22]:
numeric_features = ['Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Promotion', 'Competitor Pricing', 'Epidemic', 'Demand']


In [ ]:
#Prints out the numeric correlation matrix
numeric_corr = df.corr(numeric_only=True)
numeric_corr

,Inventory Level,Units Sold,Units Ordered,Price,Discount,Promotion,Competitor Pricing,Epidemic,Demand
Inventory Level,1.000000,0.234590,-0.034698,-0.036673,-0.000678,-0.006082,-0.034754,-0.097550,0.126618
Units Sold,0.234590,1.000000,0.433943,-0.014506,0.183523,0.227062,-0.013989,-0.317855,0.833421
Units Ordered,-0.034698,0.433943,1.000000,0.044980,0.172972,0.214743,0.045235,-0.100546,0.511963
Price,-0.036673,-0.014506,0.044980,1.000000,-0.094136,-0.043190,0.976648,-0.089658,-0.023461
Discount,-0.000678,0.183523,0.172972,-0.094136,1.000000,0.784039,-0.092066,-0.003885,0.224723
Promotion,-0.006082,0.227062,0.214743,-0.043190,0.784039,1.000000,-0.041385,0.000700,0.282537
Competitor Pricing,-0.034754,-0.013989,0.045235,0.976648,-0.092066,-0.041385,1.000000,-0.089466,-0.023036
Epidemic,-0.097550,-0.317855,-0.100546,-0.089658,-0.003885,0.000700,-0.089466,1.000000,-0.363661
Demand,0.126618,0.833421,0.511963,-0.023461,0.224723,0.282537,-0.023036,-0.363661,1.000000


In [23]:
for col in numeric_features:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"{col}: " + str(len(outliers)) + " outliers")

Inventory Level: 2759 outliers
Units Sold: 1411 outliers
Units Ordered: 7524 outliers
Price: 70 outliers
Discount: 12413 outliers
Promotion: 0 outliers
Competitor Pricing: 185 outliers
Epidemic: 15200 outliers
Demand: 986 outliers


In [20]:
#Trying a Spearman Correlation Matrix since massive amount of outliers
df[numeric_features].corr(method='spearman')

,Inventory Level,Units Sold,Units Ordered,Price,Discount,Promotion,Competitor Pricing,Epidemic,Demand
Inventory Level,1.000000,0.252028,-0.277970,0.007980,-0.000829,-0.005777,0.010174,-0.106334,0.117184
Units Sold,0.252028,1.000000,0.337778,-0.013064,0.166772,0.211554,-0.012589,-0.341171,0.847304
Units Ordered,-0.277970,0.337778,1.000000,0.024945,0.146276,0.189399,0.024087,-0.075258,0.426569
Price,0.007980,-0.013064,0.024945,1.000000,-0.083128,-0.039252,0.983696,-0.081835,-0.024321
Discount,-0.000829,0.166772,0.146276,-0.083128,1.000000,0.765674,-0.080804,-0.003444,0.204341
Promotion,-0.005777,0.211554,0.189399,-0.039252,0.765674,1.000000,-0.037759,0.000700,0.261325
Competitor Pricing,0.010174,-0.012589,0.024087,0.983696,-0.080804,-0.037759,1.000000,-0.080965,-0.024071
Epidemic,-0.106334,-0.341171,-0.075258,-0.081835,-0.003444,0.000700,-0.080965,1.000000,-0.376951
Demand,0.117184,0.847304,0.426569,-0.024321,0.204341,0.261325,-0.024071,-0.376951,1.000000


In [ ]:
#Checks to see how many null values there are
for col in df.columns:
    null_count = df[col].isnull().sum()
    print(f"{col} has {null_count} null values")
    

Date has 0 null values
Store ID has 0 null values
Product ID has 0 null values
Category has 0 null values
Region has 0 null values
Inventory Level has 0 null values
Units Sold has 0 null values
Units Ordered has 0 null values
Price has 0 null values
Discount has 0 null values
Weather Condition has 0 null values
Promotion has 0 null values
Competitor Pricing has 0 null values
Seasonality has 0 null values
Epidemic has 0 null values
Demand has 0 null values


In [54]:
#Converts the Date columns type to Datetime
df['Date'] = pd.to_datetime(df['Date'])

#Feature Engineering
df['PriceToCompetitorPriceRatio'] = df['Price'] / df['Competitor Pricing']
df['UnitsSoldToUnitsOrderedRatio'] = df['Units Sold'] / df['Units Ordered']
df['PromotionToUnitsSold'] = df['Promotion'] / df['Units Sold']
df['DiscountToUnitsSold'] = df['Discount'] / df['Units Sold']
df['UnitsSoldToInventoryLevel'] = df['Units Sold'] / df['Inventory Level']

df['PriceToCompetitorPriceRatio'] = df['PriceToCompetitorPriceRatio'].fillna(0)
df['UnitsSoldToUnitsOrderedRatio'] = df['UnitsSoldToUnitsOrderedRatio'].fillna(0)
df['PromotionToUnitsSold'] = df['PromotionToUnitsSold'].fillna(0)
df['DiscountToUnitsSold'] = df['DiscountToUnitsSold'].fillna(0)
df['UnitsSoldToInventoryLevel'] = df['UnitsSoldToInventoryLevel'].fillna(0)


#Looks at DataFrame with new features that were engineered
df.head()


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand,PriceToCompetitorPriceRatio,UnitsSoldToUnitsOrderedRatio,PromotionToUnitsSold,DiscountToUnitsSold,UnitsSoldToInventoryLevel
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,0,85.73,Winter,0,115,0.848244,0.404762,0.000000,0.049020,0.523077
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,...,1,92.02,Winter,0,229,0.871115,0.469880,0.008547,0.128205,1.000000
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,...,1,60.08,Winter,0,157,1.047603,0.186275,0.008772,0.087719,0.461538
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,...,0,85.19,Winter,0,52,1.028642,0.441176,0.000000,0.222222,0.323741
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,...,0,51.63,Winter,0,59,1.053845,0.239852,0.000000,0.000000,0.427632


In [61]:
for col in numeric_features:
    data_skew = skew(df[col])
    data_kurt = kurtosis(df[col])
    print(f"{col}: skewness is {data_skew}")
    print(f"{col}: kurtosis is {data_kurt}")
    print()

Inventory Level: skewness is 1.5919069925256135
Inventory Level: kurtosis is 3.3714100379686167

Units Sold: skewness is 0.7904129188718336
Units Sold: kurtosis is 1.2489063301175474

Units Ordered: skewness is 2.5158281864717487
Units Ordered: kurtosis is 7.305241407378645

Price: skewness is 0.4850153855444314
Price: kurtosis is -0.4839910813405406

Discount: skewness is 0.6431359608473526
Discount: kurtosis is -0.40526454067581685

Promotion: skewness is 0.7281456436945652
Promotion: kurtosis is -1.4698039215686274

Competitor Pricing: skewness is 0.548992432992015
Competitor Pricing: kurtosis is -0.30975087241995025

Epidemic: skewness is 1.4999999999999998
Epidemic: kurtosis is 0.24999999999999956

Demand: skewness is 0.6077655770166355
Demand: kurtosis is 0.6748137478964051



In [57]:
#Prints out central tendency and standard deviation
df.describe()

C:\Users\harri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
C:\Users\harri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
C:\Users\harri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
C:\Users\harri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered

,Date,Inventory Level,Units Sold,Units Ordered,Price,Discount,Promotion,Competitor Pricing,Epidemic,Demand,PriceToCompetitorPriceRatio,UnitsSoldToUnitsOrderedRatio,PromotionToUnitsSold,DiscountToUnitsSold,UnitsSoldToInventoryLevel
count,76000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,7.600000e+04,7.600000e+04,7.600000e+04,76000.000000
mean,2023-01-15 12:00:00,301.062842,88.827316,89.090645,67.726028,9.087039,0.328947,69.454029,0.200000,104.317158,0.984420,inf,inf,inf,0.435242
min,2022-01-01 00:00:00,0.000000,0.000000,0.000000,4.740000,0.000000,0.000000,4.290000,0.000000,4.000000,0.825472,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,2022-07-09 18:00:00,136.000000,58.000000,0.000000,31.997500,5.000000,0.000000,32.620000,0.000000,71.000000,0.883715,7.313433e-01,0.000000e+00,3.816794e-02,0.188885
50%,2023-01-15 12:00:00,227.000000,84.000000,0.000000,64.500000,10.000000,0.000000,65.700000,0.000000,100.000000,1.000613,NaN,0.000000e+00,9.433962e-02,0.348387
75%,2023-07-24 06:00:00,408.000000,114.000000,121.000000,95.830000,10.000000,1.000000,97.932500,0.000000,133.000000,1.080769,NaN,7.518797e-03,1.704545e-01,0.640625
max,2024-01-30 00:00:00,2267.000000,426.000000,1616.000000,228.030000,25.000000,1.000000,261.220000,1.000000,430.000000,1.177233,inf,inf,inf,1.000000
std,NaN,226.510161,43.994525,162.404627,39.377899,7.475781,0.469834,40.943818,0.400003,46.964801,0.108315,NaN,NaN,NaN,0.298917


In [63]:
#Prints out the median for the numeric columns
for col in numeric_features:
    median = df[col].median()
    std = df[col].std()
    var = df[col].var()
    print(f"{col} median: {median}")
    print(f"{col} std: {std}")
    print(f"{col} variance: {var}")
    print()

Inventory Level median: 227.0
Inventory Level std: 226.51016094087709
Inventory Level variance: 51306.85300946204

Units Sold median: 84.0
Units Sold std: 43.99452495999802
Units Sold variance: 1935.5182264558887

Units Ordered median: 0.0
Units Ordered std: 162.40462747050844
Units Ordered variance: 26375.26302383463

Price median: 64.5
Price std: 39.37789879880463
Price variance: 1550.618913808899

Discount median: 10.0
Discount std: 7.475781178320364
Discount variance: 55.88730422612901

Promotion median: 0.0
Promotion std: 0.4698339086900178
Promotion variance: 0.22074390175493996

Competitor Pricing median: 65.7
Competitor Pricing std: 40.9438175304619
Competitor Pricing variance: 1676.3961939677592

Epidemic median: 0.0
Epidemic std: 0.4000026316049173
Epidemic variance: 0.16000210529085915

Demand median: 100.0
Demand std: 46.96480105784763
Demand variance: 2205.692538403206



In [ ]:
#Shows the unique values of each of the categorical variables
cardinality_all = df.select_dtypes(exclude='number').nunique()
cardinality_all

Date                 760
Store ID               5
Product ID            20
Category               5
Region                 4
Weather Condition      4
Seasonality            4
dtype: int64